# **Generative Music with a Transformer (MIDI Generation)**

In this notebook we will train a decoder-only Transformer to generate piano music. The approach is the same next-token prediction we used for text generation — we tokenize a sequence, train a Transformer to predict the next token, and then generate new sequences autoregressively. The only difference is that instead of text tokens, we work with **MIDI tokens** that represent musical events (which note to play, how hard to press it, when to release it, and how long to wait between events).

This notebook keeps things as simple as possible — **no custom layers or subclassing**. We build the entire model using native Keras layers and the Functional API.

# *Install Dependencies and Import Libraries*

In [ ]:
!pip install -qq midi_neural_processor
!pip install -qq "keras>=3.6.0"

# For audio playback
!sudo apt-get -qq install -y fluidsynth 2> /dev/null
!pip install -qq pyfluidsynth scipy pretty_midi

In [ ]:
import os
import random
import tempfile

import keras
import midi_neural_processor.processor as midi_tokenizer
import numpy as np
from keras import layers, ops, optimizers, callbacks, utils
from os import path

# *Understanding MIDI Tokenization*

MIDI (Musical Instrument Digital Interface) files don't store audio waveforms — they store **events** that describe what a musician does. Think of it like sheet music in digital form. The `midi_neural_processor` library converts these events into integer tokens, just like we converted characters or words into integers in our text models.

There are four types of MIDI events, each mapped to a range of integer values:

| Event Type | What It Means | Example |
|-----------|--------------|--------|
| **Note On** (0–127) | Start playing a note (128 possible pitches, from very low to very high) | Token 60 = start playing Middle C |
| **Note Off** (128–255) | Stop playing a note | Token 188 = stop playing Middle C |
| **Time Shift** (256–355) | Wait before the next event (in 10ms increments, up to 1 second) | Token 306 = wait 500ms |
| **Velocity** (356–387) | How hard the next note is struck (32 levels of loudness) | Token 370 = medium-loud |

A short melody like playing Middle C for half a second, then E for half a second, might look like:

```
Token 370  → Set velocity (medium)
Token 60   → Note On: Middle C
Token 306  → Wait 500ms
Token 188  → Note Off: Middle C
Token 64   → Note On: E
Token 306  → Wait 500ms
Token 192  → Note Off: E
```

This is a **discrete sequence of integers** — exactly what a Transformer is designed to model. The same next-token prediction approach that generates text can generate music.

# *Configuration*

In [ ]:
# Total count of possible MIDI event tokens
event_range = midi_tokenizer.RANGE_NOTE_ON
event_range += midi_tokenizer.RANGE_NOTE_OFF
event_range += midi_tokenizer.RANGE_TIME_SHIFT
event_range += midi_tokenizer.RANGE_VEL

# We add 3 special tokens on top of the MIDI events: padding, start-of-sequence, end-of-sequence
VOCAB_SIZE       = event_range + 3
TOKEN_PAD        = event_range
TOKEN_START      = event_range + 1
TOKEN_END        = event_range + 2

MAX_SEQ_LEN      = 2048
EMBED_DIM        = 256
NUM_HEADS        = 4
FF_DIM           = 512
NUM_LAYERS       = 6
DROPOUT          = 0.2
BATCH_SIZE       = 6
SEED             = 42
MODEL_OUT        = "tmp/music_transformer_simple.keras"

utils.set_random_seed(SEED)

print(f'MIDI event tokens: {event_range}, total vocabulary (with special tokens): {VOCAB_SIZE}')
print(f'Max sequence length: {MAX_SEQ_LEN}')

# *Download and Preprocess the Maestro Dataset*

We use the **Maestro dataset** — ~1,200 classical piano performances from international piano competitions.

In [ ]:
def download_maestro(output_dir=None):
    """Download the Maestro MIDI dataset."""
    output_dir = tempfile.mkdtemp() if output_dir is None else output_dir
    os.makedirs(output_dir, exist_ok=True)

    dir = utils.get_file(
        origin="https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip",
        extract=True,
    )

    midi_files, file_paths = set(), list()
    for root, _, files in os.walk(dir):
        for file in files:
            if file.lower().endswith(".midi") or file.lower().endswith(".mid"):
                midi_files.add(path.join(root, file))

    for file in sorted(midi_files):
        file_paths.append(new_path := path.join(output_dir, path.basename(file)))
        os.rename(file, new_path)
    return file_paths

paths = list(sorted(download_maestro(output_dir="datasets/maestro")))
output_dir = path.dirname(paths[0])

print(f'Downloaded {len(paths)} MIDI files.')

In [ ]:
# Train/validation split (90/10)
indices = np.random.permutation(len(paths))
split = int(len(paths) * 0.1)
train_paths = [paths[i] for i in indices[split:]]
val_paths = [paths[i] for i in indices[:split]]

print(f'Training files: {len(train_paths)}, Validation files: {len(val_paths)}')

# *Listen to a Training Sample*

In [ ]:
import pretty_midi
from scipy.io.wavfile import write as write_wav
from IPython.display import Audio

def visualize_midi(midi_path, sampling_rate=16000, seconds=15, out_dir=None):
    """Convert a MIDI file to audio for playback."""
    pretty_midi_file = pretty_midi.PrettyMIDI(midi_path)
    waveform = pretty_midi_file.fluidsynth(fs=sampling_rate)[: seconds * sampling_rate]

    if out_dir is None:
        return Audio(waveform, rate=sampling_rate)

    os.makedirs(out_dir, exist_ok=True)
    audio_path = path.join(out_dir, path.basename(midi_path).split(".")[0] + ".wav")
    write_wav(audio_path, sampling_rate, (waveform * 32767).astype(np.int16))
    return audio_path

# Listen to the first 15 seconds of a training sample
visualize_midi(train_paths[0])

# *Tokenize and Build the Dataset*

We tokenize every MIDI file into integer sequences using `midi_neural_processor`. This step may take a few minutes on the first run, but the results are cached to disk.

In [ ]:
def encode_midi_task(midi_path):
    """Tokenize a single MIDI file (used by multiprocessing pool)."""
    import midi_neural_processor.processor as midi_tokenizer
    return midi_tokenizer.encode_midi(midi_path)

def preprocess_midi_files(file_paths, save_dir=None):
    """Preprocess a list of MIDI files and cache the results."""
    from multiprocessing import Pool, cpu_count

    save_dir = path.dirname(file_paths[0]) if save_dir is None else save_dir
    os.makedirs(save_dir, exist_ok=True)

    # Check if already preprocessed
    output_file = path.join(save_dir, "notes.npz")
    if path.exists(output_file):
        npz_file = np.load(output_file)
        return [npz_file[key] for key in npz_file.keys()]

    # Preprocess in parallel
    progbar = utils.Progbar(len(file_paths), unit_name="MIDI_file", interval=5)
    pool = Pool(cpu_count() - 1)
    all_notes = []
    for notes in pool.imap_unordered(encode_midi_task, file_paths):
        progbar.add(1)
        all_notes.append(np.array(notes))

    np.savez(output_file, *all_notes)
    return all_notes

train_midis = preprocess_midi_files(train_paths, path.join(output_dir, "train"))
val_midis = preprocess_midi_files(val_paths, path.join(output_dir, "val"))

print(f'Tokenized {len(train_midis)} training files and {len(val_midis)} validation files.')
print(f'Example: first training file has {len(train_midis[0])} tokens.')

Each training example is a pair of sequences offset by one token: the model sees tokens 0 through N-1 as input and predicts tokens 1 through N as output.

In [ ]:
class MidiDataset(utils.PyDataset):
    """Yields batches of (input_sequence, target_sequence) from tokenized MIDI."""

    def __init__(self, encoded_midis, batch_size=BATCH_SIZE, max_seq_len=MAX_SEQ_LEN):
        super(MidiDataset, self).__init__()
        self.batch_size = batch_size
        self.max_seq_len = max_seq_len
        self.encoded_midis = encoded_midis
        batches, remainder = divmod(len(encoded_midis), batch_size)
        self._num_batches = batches + int(remainder > 0)

    def __len__(self):
        return self._num_batches

    def __getitem__(self, idx):
        batch = random.sample(self.encoded_midis, k=self.batch_size)
        batch_data = np.array([self._get_sequence(midi) for midi in batch])

        # Input is tokens 0..N-1, target is tokens 1..N
        return batch_data[:, :-1], batch_data[:, 1:]

    def _get_sequence(self, data):
        """Extract a random subsequence and pad if needed."""
        length = self.max_seq_len + 1  # +1 because we split into input/target

        if len(data) > length:
            start = random.randrange(0, len(data) - length)
            data = data[start : start + length]
        elif len(data) < length:
            data = np.append(data, TOKEN_END)

        if len(data) < length:
            data = np.concatenate((data, np.full(length - len(data), TOKEN_PAD)))

        return np.asanyarray(data, dtype="int32")

train_dataset = MidiDataset(train_midis)
val_dataset = MidiDataset(val_midis)

print(f'Training batches: {len(train_dataset)}, Validation batches: {len(val_dataset)}')

# *Building the Transformer*

We build the entire model using Keras's **Functional API** — just `layers.Something()(x)` calls chained together. No custom `Layer` or `Model` subclasses.

The architecture is a standard decoder-only Transformer:
1. **Token Embedding** — converts each integer token to a dense vector
2. **Positional Embedding** — learned embeddings so the model knows token order
3. **N Transformer Decoder Blocks**, each containing:
   - **Causal Multi-Head Self-Attention** — each token attends only to previous tokens (the `use_causal_mask=True` flag handles the masking automatically)
   - **Feed-Forward Network** — two Dense layers with ReLU activation
   - **Layer Normalization** and **Residual Connections** around each sub-layer
4. **Output Dense layer** — projects back to vocabulary size for next-token prediction

In [ ]:
def build_music_transformer(
    vocab_size=VOCAB_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
):
    """Build a decoder-only Transformer using only native Keras layers."""

    inputs = layers.Input(shape=(max_seq_len,), dtype="int32", name="token_input")

    # --- Embedding ---
    # Token embedding: convert integer tokens to dense vectors
    token_emb = layers.Embedding(vocab_size, embed_dim, name="token_embedding")(inputs)

    # Positional embedding: learned position vectors (one per sequence position)
    pos_emb = layers.Embedding(max_seq_len, embed_dim, name="position_embedding")(
        ops.arange(0, max_seq_len)
    )

    # Combine token + position embeddings
    x = token_emb + pos_emb
    x = layers.Dropout(dropout)(x)

    # --- Transformer Decoder Blocks ---
    for i in range(num_layers):
        # ---- Causal Multi-Head Self-Attention ----
        attn_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout,
            name=f"attention_{i}",
        )(query=x, value=x, key=x, use_causal_mask=True)

        attn_output = layers.Dropout(dropout)(attn_output)
        x = layers.LayerNormalization(epsilon=1e-6, name=f"norm1_{i}")(x + attn_output)

        # ---- Feed-Forward Network ----
        ffn_output = layers.Dense(ff_dim, activation="relu", name=f"ffn_relu_{i}")(x)
        ffn_output = layers.Dense(embed_dim, name=f"ffn_linear_{i}")(ffn_output)
        ffn_output = layers.Dropout(dropout)(ffn_output)
        x = layers.LayerNormalization(epsilon=1e-6, name=f"norm2_{i}")(x + ffn_output)

    # --- Output projection ---
    outputs = layers.Dense(vocab_size, name="output")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="music_transformer")
    return model

model = build_music_transformer()
model.summary()

Notice how simple this is compared to the original notebook. The key insight: Keras's built-in `MultiHeadAttention` layer has a `use_causal_mask=True` argument that automatically creates the lower-triangular mask we need for autoregressive generation. No manual mask creation required!

# *Training*

We use a learning rate warmup schedule — start with a very small learning rate and gradually increase it to avoid exploding gradients early on.

In [ ]:
def masked_loss(y_true, y_pred):
    """Cross-entropy loss that ignores padding tokens."""
    mask = ops.cast(ops.not_equal(y_true, TOKEN_PAD), "float32")
    loss = ops.categorical_crossentropy(
        ops.one_hot(ops.cast(y_true, "int32"), VOCAB_SIZE),
        y_pred,
        from_logits=True, # This means, apply the softmax before doing the loss calculation. We do it here instead of the output layer.
    )
    return loss * mask


@keras.utils.register_keras_serializable()
class WarmupSchedule(optimizers.schedules.LearningRateSchedule):
    """Learning rate warmup then decay (same idea as our Mini GPT notebook)."""

    def __init__(self, embed_dim, warmup_steps=4000):
        super().__init__()
        self.embed_dim = embed_dim
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = ops.cast(step, "float32")
        d = ops.cast(self.embed_dim, "float32")
        return ops.rsqrt(d) * ops.minimum(ops.rsqrt(step), step * (self.warmup_steps ** -1.5))

    def get_config(self):
        return {"embed_dim": self.embed_dim, "warmup_steps": self.warmup_steps}

In [ ]:
lr_schedule = WarmupSchedule(EMBED_DIM)
optimizer = optimizers.Adam(lr_schedule, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

model.compile(optimizer=optimizer, loss=masked_loss)

save_cb = callbacks.ModelCheckpoint(MODEL_OUT, save_best_only=True)

if path.exists(MODEL_OUT):
    model = keras.models.load_model(MODEL_OUT)
    print("Loaded saved model.")
else:
    model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=15,
        callbacks=[save_cb],
        verbose=2,
    )

# *Generate Music!*

Now we write a simple generation loop. We seed the model with a few tokens from a real MIDI file, then repeatedly:
1. Feed the sequence so far into the model
2. Get the predicted probabilities for the next token
3. Sample from the top-k most likely tokens (not greedy — this adds variety)
4. Append the sampled token and repeat

This is the same autoregressive generation approach we used for text.

In [ ]:
def generate(model, seed_tokens, length=1024, top_k=10):
    """
    Autoregressively generate MIDI tokens using top-k sampling.

    Args:
        model: trained Keras model
        seed_tokens: list of integer tokens to start from
        length: how many new tokens to generate
        top_k: sample from the top-k most likely next tokens
    """
    generated = list(seed_tokens)
    progbar = utils.Progbar(length, unit_name="token", interval=5)

    for _ in range(length):
        # Pad or truncate to MAX_SEQ_LEN for the model input
        current = generated[-MAX_SEQ_LEN:]  # keep only the last MAX_SEQ_LEN tokens
        padded = current + [TOKEN_PAD] * (MAX_SEQ_LEN - len(current))
        input_array = np.array([padded], dtype="int32")

        # Get model predictions (logits) for the last real token position
        logits = model.predict(input_array, verbose=0)[0]  # shape: (MAX_SEQ_LEN, VOCAB_SIZE)
        next_token_logits = logits[len(current) - 1]        # logits at the last real position

        # Top-k sampling: pick from the k most likely tokens
        top_k_indices = np.argsort(next_token_logits)[-top_k:]
        top_k_logits = next_token_logits[top_k_indices]

        # Convert logits to probabilities with softmax
        top_k_probs = np.exp(top_k_logits - np.max(top_k_logits))
        top_k_probs = top_k_probs / top_k_probs.sum()

        # Sample one token
        chosen = np.random.choice(top_k_indices, p=top_k_probs)
        generated.append(int(chosen))
        progbar.add(1)

    return generated

In [ ]:
def generate_music(model, seed_path, length=1024, top_k=15, out_dir=None):
    """Generate music from a seed MIDI file."""
    out_dir = out_dir if out_dir is not None else tempfile.mkdtemp()
    os.makedirs(out_dir, exist_ok=True)

    # Take 25 tokens from the middle of the seed file as our starting prompt
    seed = midi_tokenizer.encode_midi(seed_path)[100:125]
    print(f"Seed tokens: {seed}")

    # Generate!
    result = generate(model, seed, length=length, top_k=top_k)

    output_path = path.join(out_dir, path.basename(seed_path).split(".")[0] + ".mid")
    midi_tokenizer.decode_midi(result, output_path)
    return output_path

output_file = generate_music(model, val_paths[-1], out_dir="tmp/", top_k=15)
print(f"Generated MIDI saved to: {output_file}")

Listen to the generated music...

In [ ]:
visualize_midi(output_file)

Try with a different seed and smaller top-k (less randomness, more conservative output)...

In [ ]:
output_file_2 = generate_music(model, val_paths[0], out_dir="tmp/", top_k=5)
visualize_midi(output_file_2)

# *Discussion*

### What We Built
A decoder-only Transformer that generates piano music token by token — the same autoregressive approach we used for text generation, applied to MIDI events instead of words.

### Simplifications vs. the Full Version
The original Music Transformer paper uses **relative positional attention** (encoding the *distance* between tokens rather than their absolute positions). Our simplified version uses standard **learned positional embeddings** instead. Relative attention is better for music (a melody should sound the same regardless of when it starts), but learned positional embeddings work well enough to demonstrate the concept.

### Key Takeaways
- **Tokenization is the bridge**: just as we tokenized text into integers for language models, we tokenized musical events into integers here. The Transformer doesn't "know" it's generating music — it's just predicting the next integer in a sequence.
- **The model is just standard Keras layers**: `Embedding`, `MultiHeadAttention`, `Dense`, `LayerNormalization`, and `Dropout` — nothing custom.
- **`use_causal_mask=True`** on `MultiHeadAttention` automatically handles the lower-triangular mask that prevents the model from looking at future tokens.
- **Top-k sampling** produces more interesting output than greedy decoding. The choice of k controls the creativity-vs-coherence trade-off.